In [ ]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd
from pprint import pprint
from IPython.display import Markdown, display

_here = Path.cwd().resolve()
_repo_root = next(path for path in (_here, *_here.parents) if (path / "configs" / "cfg.py").exists())
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from data.share_data import load_share_data
from scripts.mpc import plot_mpc_outputs, run_admm_mpc, run_local_mpc
from utils.run_artifacts import load_experiment_context


In [ ]:
run_dir = Path("artifacts/runs/20260504_234339_01e73bd1")
cfg, run_dir = load_experiment_context(run_dir)
share_data = load_share_data(run_dir / "share_data", cfg)
display(Markdown("# MADRL_ESS MPC"))
display(pd.DataFrame([{"run_dir": str(run_dir), "eval_start": cfg.data.eval_start_date, "eval_end": cfg.data.eval_end_date, "eval_episodes": int(share_data.eval["price"].shape[0])}]))


In [ ]:
local_records = run_local_mpc(cfg, run_dir, share_data)
admm_record = run_admm_mpc(cfg, run_dir, share_data)
records = {"local": local_records, "admm": admm_record}
metrics_df = pd.concat([local_records["perfect"]["metrics_df"], local_records["lstm"]["metrics_df"], admm_record["metrics_df"]], ignore_index=True)
display(Markdown("## Metrics"))
display(metrics_df)
pprint({"record_root": str(run_dir / "results")})


In [ ]:
figures = plot_mpc_outputs(records, run_dir)
_plot_specs = [
    ("1. 电力电量平衡", "mpc_power_balance"),
    ("2. 电价", "mpc_price"),
    ("3. 总体储能充放功率与 SoC", "mpc_battery_soc"),
    ("4. 电压", "mpc_voltage"),
    ("5. 净负荷", "mpc_net_load"),
]
for title, name in _plot_specs:
    print(title)
    display(figures[name])
    plt.close(figures[name])
